# ReAct Pattern: Reasoning + Acting

## Overview

This notebook explores the ReAct (Reasoning and Acting) pattern, where agents alternate between reasoning about problems and taking actions to solve them.

## Learning Objectives

- Understand the ReAct pattern and its components
- Implement ReAct agents with LangChain
- Create and integrate tools for agent actions
- Handle errors and prevent infinite loops
- Analyze ReAct execution traces

## Exam Objectives: 1.2, 1.6
## Estimated Time: 60-75 minutes

## 🎯 Exam Focus

This notebook covers concepts that appear frequently on the NCP-AAI exam:

**High-Priority Topics:**
- ⭐⭐⭐ Architecture selection (reactive vs deliberative vs hybrid)
- ⭐⭐⭐ Trade-offs between architectures
- ⭐⭐ When to use each architecture pattern
- ⭐⭐ Scalability considerations

**Common Exam Scenarios:**
- Choosing architecture based on latency requirements
- Selecting architecture for different complexity levels
- Balancing speed vs sophistication

**Key Concepts to Master:**
- Reactive: Fast, rule-based, limited flexibility
- Deliberative: Complex reasoning, slower, goal-oriented
- Hybrid: Production-ready, balances both approaches

## Setup: Import Dependencies

In [ ]:
# Import core libraries
import os
from typing import Dict, List, Any

# LangChain imports for ReAct agents
try:
    from langchain.agents import Tool, AgentExecutor, create_react_agent
    from langchain.prompts import PromptTemplate
    print("✅ LangChain agent modules loaded")
except ImportError as e:
    print(f"❌ Error loading LangChain: {e}")

print("\n🎯 Setup complete!")

## Theory: The ReAct Pattern

### What is ReAct?

**ReAct** (Reasoning and Acting) is a paradigm where agents alternate between:

1. **Thought** - Reasoning about the problem
2. **Action** - Taking an action (calling a tool)
3. **Observation** - Receiving feedback from the action
4. **Repeat** - Iterating until goal is achieved

### ReAct Loop Structure

```
User Query → Thought → Action → Observation → Thought → ... → Answer
```

### Example Trace

```
Question: What is the population of Tokyo multiplied by 2?

Thought: I need to find the population of Tokyo first.
Action: Search("population of Tokyo")
Observation: Tokyo has approximately 14 million people.

Thought: Now I need to multiply this by 2.
Action: Calculator("14000000 * 2")
Observation: Result: 28000000

Thought: I now have the final answer.
Answer: The population of Tokyo multiplied by 2 is 28,000,000.
```

## Implementation: Creating Tools

Tools are functions that agents can call to interact with external systems.

In [ ]:
# Define tools for the ReAct agent

def search_tool(query: str) -> str:
    """
    Simulate a search tool that finds information.
    
    Args:
        query: Search query string
    
    Returns:
        Search results as string
    """
    # Simulated search results for demonstration
    search_db = {
        "population of tokyo": "Tokyo has a population of approximately 14 million people in the city proper and 37 million in the greater metropolitan area.",
        "capital of france": "The capital of France is Paris.",
        "nvidia founded": "NVIDIA was founded in 1993 by Jensen Huang, Chris Malachowsky, and Curtis Priem.",
        "langchain": "LangChain is a framework for developing applications powered by language models."
    }
    
    # Search for matching results
    query_lower = query.lower()
    for key, value in search_db.items():
        if key in query_lower:
            return value
    
    return f"No results found for: {query}"

def calculator_tool(expression: str) -> str:
    """
    Perform mathematical calculations.
    
    Args:
        expression: Mathematical expression to evaluate
    
    Returns:
        Calculation result as string
    """
    try:
        # Evaluate mathematical expression safely
        # Note: In production, use a safer evaluation method
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception as e:
        # Handle calculation errors gracefully
        return f"Error in calculation: {e}"

def get_current_date_tool(input: str = "") -> str:
    """
    Get the current date.
    
    Args:
        input: Unused parameter (required by tool interface)
    
    Returns:
        Current date as string
    """
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d")

# Create Tool objects for LangChain
tools = [
    Tool(
        name="Search",
        func=search_tool,
        description="Useful for finding information about topics, places, people, or facts. Input should be a search query."
    ),
    Tool(
        name="Calculator",
        func=calculator_tool,
        description="Useful for performing mathematical calculations. Input should be a mathematical expression like '2 + 2' or '100 * 0.15'."
    ),
    Tool(
        name="GetDate",
        func=get_current_date_tool,
        description="Useful for getting the current date. No input required."
    )
]

# Test tools individually
print("Testing Tools:\n")
print("Search:", search_tool("population of Tokyo"))
print("Calculator:", calculator_tool("14000000 * 2"))
print("GetDate:", get_current_date_tool())

## Implementation: Simple ReAct Agent (Without LLM)

Let's implement a simplified ReAct agent to understand the pattern.

In [ ]:
# Simplified ReAct agent implementation
class SimpleReActAgent:
    """Simplified ReAct agent for demonstration purposes."""
    
    def __init__(self, tools: List[Tool], max_iterations: int = 5):
        """
        Initialize ReAct agent.
        
        Args:
            tools: List of available tools
            max_iterations: Maximum number of reasoning-action cycles
        """
        self.tools = {tool.name: tool for tool in tools}
        self.max_iterations = max_iterations
        self.trace = []  # Store execution trace
    
    def think(self, query: str, observations: List[str]) -> Dict[str, str]:
        """
        Reasoning step: decide what action to take.
        
        Args:
            query: Original user query
            observations: Previous observations from actions
        
        Returns:
            Dictionary with thought and action
        """
        # Simplified reasoning logic (in real ReAct, LLM does this)
        if not observations:
            # First iteration - analyze query
            if "population" in query.lower():
                return {
                    "thought": "I need to search for population information.",
                    "action": "Search",
                    "action_input": query
                }
            elif any(op in query for op in ['+', '-', '*', '/', 'calculate']):
                return {
                    "thought": "I need to perform a calculation.",
                    "action": "Calculator",
                    "action_input": query
                }
        else:
            # Subsequent iterations - check if we have enough info
            if "multiply" in query.lower() and len(observations) == 1:
                return {
                    "thought": "I have the population, now I need to multiply it.",
                    "action": "Calculator",
                    "action_input": "14000000 * 2"
                }
            else:
                return {
                    "thought": "I have enough information to answer.",
                    "action": "Final Answer",
                    "action_input": observations[-1]
                }
        
        return {
            "thought": "I have the answer.",
            "action": "Final Answer",
            "action_input": observations[-1] if observations else "I don't know."
        }
    
    def act(self, action: str, action_input: str) -> str:
        """
        Execute an action using the appropriate tool.
        
        Args:
            action: Name of the tool to use
            action_input: Input for the tool
        
        Returns:
            Observation from executing the action
        """
        if action == "Final Answer":
            return action_input
        
        # Execute tool and return observation
        if action in self.tools:
            try:
                result = self.tools[action].func(action_input)
                return result
            except Exception as e:
                # Handle tool execution errors
                return f"Error executing {action}: {e}"
        else:
            return f"Unknown action: {action}"
    
    def run(self, query: str) -> str:
        """
        Main ReAct loop: think, act, observe, repeat.
        
        Args:
            query: User query to process
        
        Returns:
            Final answer
        """
        observations = []
        self.trace = []
        
        print(f"Query: {query}\n")
        
        # ReAct loop with iteration limit
        for i in range(self.max_iterations):
            # Reasoning step
            decision = self.think(query, observations)
            
            # Log thought
            print(f"Thought {i+1}: {decision['thought']}")
            self.trace.append({"type": "thought", "content": decision['thought']})
            
            # Check if we're done
            if decision['action'] == "Final Answer":
                print(f"\nFinal Answer: {decision['action_input']}")
                return decision['action_input']
            
            # Action step
            print(f"Action {i+1}: {decision['action']}({decision['action_input']})")
            self.trace.append({"type": "action", "content": f"{decision['action']}({decision['action_input']})"})
            
            # Observation step
            observation = self.act(decision['action'], decision['action_input'])
            print(f"Observation {i+1}: {observation}\n")
            self.trace.append({"type": "observation", "content": observation})
            
            observations.append(observation)
        
        # Max iterations reached
        return "Maximum iterations reached without finding answer."

# Test the simple ReAct agent
agent = SimpleReActAgent(tools, max_iterations=5)
result = agent.run("What is the population of Tokyo multiplied by 2?")

## Exercise: Implement Error Handling

**Objective**: Add error handling to prevent infinite loops and handle tool failures.

Modify the ReAct agent to:
1. Detect when the same action is repeated
2. Handle tool execution failures gracefully
3. Provide fallback responses

In [ ]:
# Exercise: Enhanced ReAct agent with error handling
class EnhancedReActAgent(SimpleReActAgent):
    """ReAct agent with improved error handling."""
    
    def __init__(self, tools: List[Tool], max_iterations: int = 5):
        """Initialize enhanced agent."""
        super().__init__(tools, max_iterations)
        # Track previous actions to detect loops
        self.action_history = []
    
    def detect_loop(self, action: str, action_input: str) -> bool:
        """
        Detect if agent is stuck in a loop.
        
        Args:
            action: Current action
            action_input: Current action input
        
        Returns:
            True if loop detected, False otherwise
        """
        # Check if same action repeated multiple times
        current_action = (action, action_input)
        
        # Count occurrences of this action
        count = self.action_history.count(current_action)
        
        # If action repeated 3+ times, it's a loop
        if count >= 2:
            print("⚠️  Loop detected! Breaking out...")
            return True
        
        return False
    
    def run(self, query: str) -> str:
        """
        Enhanced run method with loop detection.
        
        Args:
            query: User query
        
        Returns:
            Final answer or error message
        """
        observations = []
        self.trace = []
        self.action_history = []
        
        print(f"Query: {query}\n")
        
        for i in range(self.max_iterations):
            try:
                # Reasoning step
                decision = self.think(query, observations)
                
                print(f"Thought {i+1}: {decision['thought']}")
                
                if decision['action'] == "Final Answer":
                    print(f"\nFinal Answer: {decision['action_input']}")
                    return decision['action_input']
                
                # Check for loops before acting
                if self.detect_loop(decision['action'], decision['action_input']):
                    return "Unable to complete task - agent stuck in loop."
                
                # Record action
                self.action_history.append((decision['action'], decision['action_input']))
                
                print(f"Action {i+1}: {decision['action']}({decision['action_input']})")
                
                # Execute action with error handling
                observation = self.act(decision['action'], decision['action_input'])
                print(f"Observation {i+1}: {observation}\n")
                
                observations.append(observation)
                
            except Exception as e:
                # Handle unexpected errors gracefully
                print(f"❌ Error in iteration {i+1}: {e}")
                return f"Error occurred: {e}"
        
        return "Maximum iterations reached without finding answer."

# Test enhanced agent
enhanced_agent = EnhancedReActAgent(tools, max_iterations=5)
result = enhanced_agent.run("What is the capital of France?")

## Checkpoint: Self-Assessment

Before proceeding, ensure you understand:

1. ✅ What does ReAct stand for?
   - Reasoning and Acting

2. ✅ What are the three main steps in the ReAct loop?
   - Thought (reasoning), Action (tool call), Observation (result)

3. ✅ Why do we need max_iterations?
   - To prevent infinite loops

4. ✅ What is a tool in ReAct?
   - A function the agent can call to interact with external systems

5. ✅ When should you use ReAct pattern?
   - Multi-step tasks requiring external information and tool calls

## Next Steps

Continue to:
- **03-multi-agent-systems.ipynb**: Learn about multi-agent coordination

## References

- Course Notes: module-01-agent-architecture-design.md (Section 3)
- LangChain ReAct Documentation
- ReAct Paper: Yao et al., 2022